# Feature Creation

This notebook creates business-oriented features by integrating multiple Olist datasets.

The engineered features will provide meaningful business insights and improve the performance of downstream Machine Learning models.

In [1]:
import pandas as pd
from pathlib import Path

## Define Dataset Location

Load the cleaned datasets generated during the data cleaning phase.

In [2]:
INTERIM_DATA_DIR = Path("../data/interim/olist")

## Load Required Datasets

Only the datasets required for feature creation are loaded to reduce memory usage and improve readability.

In [3]:
customers = pd.read_csv(INTERIM_DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(INTERIM_DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(INTERIM_DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(INTERIM_DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(INTERIM_DATA_DIR / "olist_order_reviews_dataset.csv")

## Merge Business Tables

Merge related datasets to construct a unified analytical dataset suitable for feature engineering.

In [4]:
features = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(payments, on="order_id", how="left")
    .merge(reviews, on="order_id", how="left")
    .merge(order_items, on="order_id", how="left")
)

features.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22


## Create Customer Features

Generate customer-centric business metrics that describe purchasing behavior.

In [5]:
customer_features = (
    features
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("payment_value", "sum"),
        average_order_value=("payment_value", "mean"),
        average_review_score=("review_score", "mean"),
    )
    .reset_index()
)

customer_features.head()

,customer_unique_id,total_orders,total_spent,average_order_value,average_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0


## Review Engineered Features

Inspect the generated feature table before additional preprocessing.

In [6]:
customer_features.describe(include="all")

,customer_unique_id,total_orders,total_spent,average_order_value,average_review_score
count,96096,96096.000000,96096.000000,96095.000000,95380.000000
unique,96096,NaN,NaN,NaN,NaN
top,0000366f3b9a7992bf8c76cfdf3221e2,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,1.034809,214.157343,158.847331,4.084696
std,NaN,0.214384,644.891987,219.651010,1.342024
min,NaN,1.000000,0.000000,0.000000,1.000000
25%,NaN,1.000000,64.000000,60.850000,4.000000
50%,NaN,1.000000,113.340000,103.830000,5.000000
75%,NaN,1.000000,203.422500,175.280000,5.000000


## Save Feature Dataset

Store the engineered feature table for use in the exploratory data analysis and Machine Learning phases.

In [7]:
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

customer_features.to_csv(
    PROCESSED_DATA_DIR / "customer_features.csv",
    index=False
)

## Conclusion

In this notebook:

- Multiple relational datasets were integrated.
- Customer-level business features were engineered.
- The feature dataset was exported for the next stages of the pipeline.

These engineered features will serve as the foundation for exploratory analysis and predictive modeling.